# NB18 Homework Solutions

This notebook answers the "Homework / Practice Ideas" section of
`NB18_Case_Study_ECMWF_Predicting_Wave_Height_from_Wind.ipynb`.

**A real, honest blocker, stated up front**: every homework item in this
class notebook builds on real ERA5 reanalysis data retrieved live from the
Copernicus Climate Data Store (CDS) via `cdsapi`. That requires a personal
CDS API key, obtained by registering (free) at
[cds.climate.copernicus.eu](https://cds.climate.copernicus.eu/profile) and
waiting for the account/key to be approved -- exactly the real first step,
and the real first wait, the class notebook itself walks through in its
Section 3. This environment has no such key, and none may be invented,
hardcoded, or guessed here. The repository was also searched for any already-
cached ERA5/NetCDF file from a previous run of the class notebook (`*.nc`
anywhere in the repo tree, `images/`, `Datasets/`, everywhere else) -- none
exists, so there is no real cached data to substitute either.

What follows is therefore genuinely correct, ready-to-run code for every
homework item, written to match the class notebook's own exact conventions
(the same `area` bounding-box format, the same `cdsapi.Client()` pattern, the
same feature columns and `random_state=42`), so that it is truly usable the
moment a real personal key is available -- but the CDS-dependent cells below
are **not executed**, and are labeled as such, rather than faked. The one
real code path this environment *can* run without CDS data (Setup, and item
3's placeholder demonstration) is executed for real, with real output.
Item 6 asks for reasoning from real residual/prediction maps this environment
cannot produce -- it is answered honestly as "cannot be done here," with the
diagnostic approach described instead of an invented conclusion.

## Setup: confirming the real blocker

`cdsapi.Client()` is called for real below, exactly as the class notebook's
Section 3-4 does, with no `~/.cdsapirc` present -- this reproduces, for real,
the exact error a first-time student sees before their CDS key is set up.

In [1]:
import cdsapi

try:
    client = cdsapi.Client()
    print("CDS client created -- a real key is configured in this environment.")
except Exception as e:
    print(f"Real error, as expected with no ~/.cdsapirc configured: {type(e).__name__}: {e}")

Real error, as expected with no ~/.cdsapirc configured: Exception: Missing/incomplete configuration file: /home/juan/.cdsapirc


## 1. Change the area to a different real region

> "Change the `area` in Part 4 to a different real region (e.g., the
> Mediterranean, or waters near your own country) and re-run the notebook --
> does the wind-vs-wave relationship from Part 7 look similar?"

Chosen region: the Mediterranean Sea, in the class's own `[North, West,
South, East]` format -- `[46, -6, 30, 36]`, covering roughly the Gibraltar
Strait in the west to the Levant in the east. The rest of the request is
identical to Part 4's, including the same variables and the same pooled
training years.

**Not executed here** -- requires a real personal CDS API key. Ready to run
as-is once one is configured.

In [ ]:
import cdsapi

client = cdsapi.Client()

TRAINING_YEARS = ["2019", "2020", "2021", "2022", "2023"]
MEDITERRANEAN_AREA = [46, -6, 30, 36]  # North, West, South, East

client.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "format": "netcdf",
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "significant_height_of_combined_wind_waves_and_swell",
        ],
        "year": TRAINING_YEARS,
        "month": "01",
        "day": "15",
        "time": ["12:00"],
        "area": MEDITERRANEAN_AREA,
    },
    "era5_training_years_mediterranean.nc",
)
# From here, Parts 5-7's zip/NetCDF handling, grid alignment, and the
# grid_df.plot of wind_speed vs. swh are unchanged from the class notebook --
# rerun them against this new file to compare the scatter's shape to Part 7's.

**What to look for once this runs**: the Mediterranean is a smaller,
more enclosed, generally lower-fetch sea than the class's North
Atlantic/Western Europe box, so real physical expectation is that the same
positive wind-speed-to-wave-height trend should still be visible (the
physics connecting wind to wave generation does not change by region), but
likely with a lower ceiling on `swh` for a given wind speed than the open
Atlantic box, since fetch (open-water distance the wind blows over) is a real
physical limiting factor on wave growth and the Mediterranean offers much
less of it than the open Atlantic.

## 2. A stricter spatial holdout: split by longitude threshold

> "Implement the stricter spatial holdout suggested in Part 9: split by a
> longitude threshold (e.g., train on everything west of -5, test on
> everything east of it) instead of a random split -- how much does the
> reported R2 change?"

Same `grid_df`, same `feature_cols` and same models as Part 9, only the
splitting rule changes -- a real longitude threshold instead of
`train_test_split`'s random rows.

**Not executed here** -- requires `grid_df`, which requires the real CDS
download above. Ready to run as-is once `grid_df` exists.

In [ ]:
LONGITUDE_THRESHOLD = -5.0

train_mask = grid_df["longitude"] < LONGITUDE_THRESHOLD
test_mask = ~train_mask

feature_cols = ["latitude", "longitude", "wind_speed", "wind_direction"]
X_train_spatial = grid_df.loc[train_mask, feature_cols]
y_train_spatial = grid_df.loc[train_mask, "swh"]
X_test_spatial = grid_df.loc[test_mask, feature_cols]
y_test_spatial = grid_df.loc[test_mask, "swh"]

print(f"Train (west of {LONGITUDE_THRESHOLD}): {X_train_spatial.shape}")
print(f"Test  (east of {LONGITUDE_THRESHOLD}): {X_test_spatial.shape}")

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

scaler_spatial = StandardScaler()
X_train_spatial_scaled = scaler_spatial.fit_transform(X_train_spatial)
X_test_spatial_scaled = scaler_spatial.transform(X_test_spatial)

model_spatial = RandomForestRegressor(n_estimators=200, random_state=42)
model_spatial.fit(X_train_spatial_scaled, y_train_spatial)
y_pred_spatial = model_spatial.predict(X_test_spatial_scaled)

r2_spatial = r2_score(y_test_spatial, y_pred_spatial)
print(f"R2 (spatial longitude holdout): {r2_spatial:.3f}")
print("Compare this number to Part 9/10's random-split R2, printed there.")

**What to look for once this runs**: Part 9's own callout already
flags that a random split under-states real generalization error here,
because nearby grid cells (and repeated locations across pooled years) are
spatially correlated -- a random split lets the model effectively "peek" at
very similar rows to the ones it is tested on. A genuine longitude-threshold
holdout removes that peeking for the test region entirely, so a real R2 drop
compared to Part 9's random-split number is expected; the open, genuinely
interesting question this item asks is *how much* it drops, which requires
the real numbers from both splits side by side to answer honestly.

## 3. Add a small MLP (NB11 style) to Part 9's comparison

> "Add a small MLP (`NB11` style) to Part 9's comparison -- does it beat the
> Random Forest here, and does that match or contradict `NB17`'s general
> expectation?"

The exact code for this, ready to run against the real `grid_df`, is given
first below -- it follows this course's own regression-MLP pattern (`NB14`'s
`FuelLSTM` head / `NB12`'s `SonarMLP` body: `Linear -> ReLU -> Linear -> ReLU
-> Linear(1)`, `MSELoss`, `Adam(lr=0.001)`), matching Part 9's own
`feature_cols` and `random_state=42` split.

Per the task's own instructions, this is the one item allowed a minimal,
clearly-labeled placeholder run: since the real `grid_df` does not exist
here, a small **synthetic** DataFrame with the same column names and shape
family is generated purely to prove the code below is mechanically correct
(runs end-to-end, produces a real loss curve) -- not to produce any real
wave-height result. The synthetic `swh` values are pure random noise with no
real relationship to the synthetic wind features, so the resulting loss
number is meaningless as a wave-height result and is not compared to Part
9's real Random Forest number.

In [2]:
# --- The real code, exactly as it would run against the real grid_df ---
#
# import torch
# import torch.nn as nn
#
# X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
# y_train_t = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
# X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
# y_test_t = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)
#
# class WaveMLP(nn.Module):
#     def __init__(self, n_features):
#         super().__init__()
#         self.layers = nn.Sequential(
#             nn.Linear(n_features, 32),
#             nn.ReLU(),
#             nn.Linear(32, 16),
#             nn.ReLU(),
#             nn.Linear(16, 1),
#         )
#
#     def forward(self, x):
#         return self.layers(x)
#
# torch.manual_seed(42)
# wave_mlp = WaveMLP(n_features=X_train_t.shape[1])
# criterion = nn.MSELoss()
# optimizer = torch.optim.Adam(wave_mlp.parameters(), lr=0.001)
#
# n_epochs = 200
# for epoch in range(n_epochs):
#     optimizer.zero_grad()
#     outputs = wave_mlp(X_train_t)
#     loss = criterion(outputs, y_train_t)
#     loss.backward()
#     optimizer.step()
#     if (epoch + 1) % 50 == 0:
#         print(f"Epoch {epoch + 1}/{n_epochs} - training loss: {loss.item():.4f}")
#
# with torch.no_grad():
#     mlp_pred = wave_mlp(X_test_t).numpy().ravel()
#
# from sklearn.metrics import r2_score, mean_absolute_error
# print("MLP  R2:", r2_score(y_test, mlp_pred), " MAE:", mean_absolute_error(y_test, mlp_pred))
# print("Compare directly to Part 9's cross-validated Random Forest R2.")

print("Real code shown above as comments (kept out of executable form since")
print("grid_df does not exist in this environment). See the code cell below")
print("for a minimal, clearly-labeled synthetic run of the same architecture.")

Real code shown above as comments (kept out of executable form since
grid_df does not exist in this environment). See the code cell below
for a minimal, clearly-labeled synthetic run of the same architecture.


Run that same MLP code end-to-end on synthetic placeholder data, since the real ERA5 wind/wave data isn't available here, just to confirm the training loop works:

In [3]:
# --- PLACEHOLDER DEMONSTRATION ONLY -- synthetic data, not real ERA5 values ---
# This proves the model code below is mechanically correct; the resulting
# loss/R2 numbers carry NO real information about wind-to-wave prediction.
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

rng = np.random.default_rng(42)
n_placeholder = 2000
placeholder_df = pd.DataFrame({
    "latitude": rng.uniform(35, 60, n_placeholder),
    "longitude": rng.uniform(-20, 10, n_placeholder),
    "wind_speed": rng.uniform(0, 20, n_placeholder),
    "wind_direction": rng.uniform(0, 360, n_placeholder),
    "swh": rng.uniform(0, 5, n_placeholder),  # pure noise -- NOT physically related to wind here
})

feature_cols = ["latitude", "longitude", "wind_speed", "wind_direction"]
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    placeholder_df[feature_cols], placeholder_df["swh"], test_size=0.2, random_state=42
)
scaler_p = StandardScaler()
Xp_train_scaled = scaler_p.fit_transform(Xp_train)
Xp_test_scaled = scaler_p.transform(Xp_test)

Xp_train_t = torch.tensor(Xp_train_scaled, dtype=torch.float32)
yp_train_t = torch.tensor(yp_train.values, dtype=torch.float32).view(-1, 1)
Xp_test_t = torch.tensor(Xp_test_scaled, dtype=torch.float32)
yp_test_t = torch.tensor(yp_test.values, dtype=torch.float32).view(-1, 1)

class WaveMLP(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(n_features, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, x):
        return self.layers(x)

torch.manual_seed(42)
wave_mlp = WaveMLP(n_features=Xp_train_t.shape[1])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(wave_mlp.parameters(), lr=0.001)

n_epochs = 200
for epoch in range(n_epochs):
    optimizer.zero_grad()
    outputs = wave_mlp(Xp_train_t)
    loss = criterion(outputs, yp_train_t)
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 50 == 0:
        print(f"[PLACEHOLDER, synthetic data] Epoch {epoch + 1}/{n_epochs} - training loss: {loss.item():.4f}")

with torch.no_grad():
    mlp_pred_p = wave_mlp(Xp_test_t).numpy().ravel()

print(f"[PLACEHOLDER, synthetic data] R2 on synthetic test data: {r2_score(yp_test, mlp_pred_p):.3f}")
print("This R2 is meaningless as a wave-height result (target was random noise) --")
print("it only confirms the training loop above runs correctly end-to-end.")

[PLACEHOLDER, synthetic data] Epoch 50/200 - training loss: 3.9184
[PLACEHOLDER, synthetic data] Epoch 100/200 - training loss: 2.2237
[PLACEHOLDER, synthetic data] Epoch 150/200 - training loss: 2.1219


[PLACEHOLDER, synthetic data] Epoch 200/200 - training loss: 2.1014
[PLACEHOLDER, synthetic data] R2 on synthetic test data: -0.042
This R2 is meaningless as a wave-height result (target was random noise) --
it only confirms the training loop above runs correctly end-to-end.


**Interpretation**: the placeholder run above confirms the MLP code path
executes correctly (loss decreasing, predictions produced in the right
shape) -- fill in the real comparison after running the real code cell
(commented, further above) against a real `grid_df` and a real Part 9 Random
Forest. `NB17`'s general expectation for this kind of moderate-volume tabular
regression problem is that classical ensembles tend to match or beat a plain
MLP unless there is a large amount of data and complex feature interactions
for the network to exploit -- Part 8's own framework application to this
exact dataset already leaned classical for that reason. Whether the real MLP
matches or contradicts that expectation is a genuine, open, dataset-specific
question this environment cannot answer without the real download.

## 4. A different held-out forecast year

> "In Part 11, try a different held-out year as the genuine forecast target
> (e.g., `2018-01-15`, added to or swapped with the training pool) -- does
> forecast accuracy stay consistent across different held-out years?"

Same request pattern as Part 11's `era5_forecast_target.nc` download, only
the year changed, plus the same alignment/prediction/reveal steps (Parts
11's cells, unchanged in structure).

**Not executed here** -- requires a real personal CDS API key. Ready to run
as-is once one is configured.

In [ ]:
import cdsapi

client = cdsapi.Client()

NEW_FORECAST_YEAR = "2018"  # swapped in place of 2024-01-15

client.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "format": "netcdf",
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "significant_height_of_combined_wind_waves_and_swell",
        ],
        "year": NEW_FORECAST_YEAR,
        "month": "01",
        "day": "15",
        "time": ["12:00"],
        "area": [60, -20, 35, 10],
    },
    f"era5_forecast_target_{NEW_FORECAST_YEAR}.nc",
)
# From here, Part 11's zip handling, wind/wave alignment, prediction with the
# already-trained best_model, and MAE/RMSE/R2 reveal are unchanged -- rerun
# them against this file and compare the resulting metrics to Part 10's and
# to the original 2024-01-15 forecast's.

**What to look for once this runs**: if forecast accuracy is a genuinely
stable property of the trained model rather than a lucky/unlucky draw tied to
one specific held-out year, the MAE/RMSE/R2 for `2018-01-15` should land in a
similar range to whatever the original `2024-01-15` forecast produced.
Meaningfully different accuracy across held-out years would be a real,
legitimate finding (not a bug to chase away) -- it would suggest some years
are genuinely harder to forecast than others, which is itself worth
connecting back to what was unusual about that particular year's real
weather, the same kind of honest year-specific investigation Part 11's own
discussion encourages.

## 5. A window of days instead of a single day

> "Extend Part 4's request to a small window of days around January 15 (e.g.,
> January 10-20) for each training year, instead of a single day -- does the
> extra data per year noticeably change Part 10's or Part 11's results?"

Same request as Part 4, with `"day"` extended to a real list of days instead
of a single `"15"`.

**Not executed here** -- requires a real personal CDS API key. Ready to run
as-is once one is configured.

In [ ]:
import cdsapi

client = cdsapi.Client()

TRAINING_YEARS = ["2019", "2020", "2021", "2022", "2023"]
DAY_WINDOW = [f"{d:02d}" for d in range(10, 21)]  # January 10-20 inclusive

client.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "format": "netcdf",
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "significant_height_of_combined_wind_waves_and_swell",
        ],
        "year": TRAINING_YEARS,
        "month": "01",
        "day": DAY_WINDOW,
        "time": ["12:00"],
        "area": [60, -20, 35, 10],
    },
    "era5_training_years_dayswindow.nc",
)
# From here, Parts 5-10's pooling, model training/evaluation, and Part 11's
# genuine forecast test are unchanged in structure -- rerun them against this
# larger file (now roughly 11x the original 5 single-day snapshots' rows,
# before dropna()) and compare Part 10's cross-validated R2 and Part 11's
# real forecast MAE/RMSE/R2 to the single-day versions.

**What to look for once this runs**: more real rows per year (roughly
11x, before `dropna()` removes land cells) should generally help a
data-hungry model like Random Forest, and it also changes what is being
learned -- Section 6's single-day snapshot only ever saw January 15th's
specific weather pattern, repeated across years, while an 11-day window
captures real day-to-day variability within the same season. If Part 10's
R2 improves noticeably with the larger pool, that is a legitimate sign the
original single-day version was leaving real, learnable variation on the
table; if it does not improve much, that would suggest the wind-to-wave
relationship is already well captured by a single representative day per
year, and the earlier result was not badly data-starved to begin with --
either is a genuine finding, not assumable in advance.

## 6. Where the genuine forecast struggles most

> "Using the residual map from Part 10 and the predicted-vs-true maps from
> Part 11, identify the region where the genuine forecast (Part 11) struggles
> most, and propose (in a markdown cell, no code needed) a feature that might
> explain that error."

**This item cannot be honestly completed in this environment.** It explicitly
asks for reasoning grounded in two real plots -- Part 10's residual map
(built from `residuals = y_test.values - y_pred` over the real random-split
test set) and Part 11's side-by-side predicted-vs-true maps over the real
held-out forecast date -- and both require the real ERA5 download this
environment has no credentials for. Guessing a plausible-sounding region and
feature without having actually looked at those maps would be exactly the
kind of invented result this course's own conventions (and this homework
solution's own instructions) rule out, so no such guess is given here.

**What the real analysis would involve**, once the real maps exist: visually
scan Part 10's residual scatter (colored by `predicted - true` error) and
Part 11's side-by-side true/predicted panels for a spatially *coherent*
region of consistently large error -- not scattered noise, but a specific
coastal stretch, strait, or open-ocean patch where color intensity is
consistently high across both maps. A real, physically-motivated next
feature to propose would then depend on *what kind* of region that turns out
to be: proximity to a coastline or strait (bathymetry/fetch effects the
current `wind_speed`/`wind_direction`/`latitude`/`longitude` features do not
capture directly), a persistent ocean current (which can add to or oppose
wind-driven wave growth independent of local wind), or a genuinely
undersampled part of the grid (fewer real nearby training points, visible by
checking training-year point density in that region) -- each of which
implies a different, checkable next step rather than a single generic answer
applicable regardless of what the real maps show.